In [ ]:
import shapefile
import geopandas as gpd
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
from matplotlib.cm import get_cmap

from scipy.ndimage import gaussian_filter1d
from sklearn.preprocessing import MinMaxScaler
from scipy.interpolate import splprep, splev
from shapely.geometry import LineString, Point
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree

import torch
from torch_geometric.data import Data
from scipy.spatial import Delaunay

import networkx as nx
from torch_geometric.utils import to_networkx
import os

from shapely.geometry import shape as shapely_shape

## HELPER FUNCTIONS

## PUSH DATASET

In [ ]:
PATH_SHP_1 = '../data/push/all.shp'
PATH_SHP_2 = '../data/push/all_p.shp'

# Load as GeoDataFrame
gdf_1 = gpd.read_file(PATH_SHP_1)
gdf_2 = gpd.read_file(PATH_SHP_2)

### INDIVIDUAL CREATION

In [ ]:
def polyline_gdf_to_graph(gdf, id_col='UID'):
    """
    Convert a GeoDataFrame of polylines to a NetworkX graph.
    
    Parameters:
        gdf : GeoDataFrame with LineString geometries
        id_col : column that contains polyline identifiers
    
    Returns:
        G : networkx.Graph
    """
    G = nx.Graph()
    
    for idx, row in gdf.iterrows():
        poly_id = row[id_col]
        geom = row.geometry
        
        if not isinstance(geom, LineString):
            continue  # skip non-LineStrings
        
        coords = list(geom.coords)
        for i, coord in enumerate(coords):
            node_id = f"{poly_id}_{i}"  # unique node id
            G.add_node(node_id, 
                       x=coord[0], y=coord[1], 
                       poly_id=poly_id, seq=i)
            
            # Add edge to previous node
            if i > 0:
                prev_node_id = f"{poly_id}_{i-1}"
                # optionally, store distance as edge attribute
                dist = ((coord[0]-coords[i-1][0])**2 + (coord[1]-coords[i-1][1])**2)**0.5
                G.add_edge(prev_node_id, node_id, distance=dist)
                
    return G


In [ ]:
graph = polyline_gdf_to_graph(gdf_1[10:11], id_col='UID')


In [ ]:
def plot_graph(G):
    """
    Plot a NetworkX graph created from polylines.
    
    Nodes are positioned using their x, y attributes.
    """
    pos = {node: (data['x'], data['y']) for node, data in G.nodes(data=True)}
    
    plt.figure(figsize=(5, 3))
    nx.draw(G, pos, node_color='#143642', with_labels=False,
            node_size=5, edge_color='#286981')
    
    plt.title("Graph from Polylines")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.axis('equal')
    plt.show()


In [ ]:
plot_graph(graph)

In [ ]:
def extract_coords(linestring):
    """Return Nx2 array of coordinates from a LineString."""
    return np.array(linestring.coords)

In [ ]:
def compute_orientations(coords):
    diffs = np.diff(coords, axis=0)
    return np.arctan2(diffs[:,1], diffs[:,0])

def compute_angles(coords):
    orientations = compute_orientations(coords)
    angles = np.diff(orientations)

    # Normalize to [-pi, pi]
    angles = (angles + np.pi) % (2*np.pi) - np.pi
    return angles

def compute_distances(coords):
    diffs = np.diff(coords, axis=0)
    return np.sqrt((diffs**2).sum(axis=1))


def compute_bending(coords):
    angles = compute_angles(coords)
    distances = compute_distances(coords)

    # curvature at vertex i = angle_i / (d_i + d_(i+1))
    bending = angles / (distances[:-1] + distances[1:])
    return bending


In [ ]:
# --- convert segment-based features to per-point arrays ---

def features_per_point(coords):
    n = len(coords)

    seg_ori = compute_orientations(coords)     # length n-1
    seg_dist = compute_distances(coords)       # length n-1
    vert_ang = compute_angles(coords)          # length n-2
    vert_bend = compute_bending(coords)        # length n-2

    # Initialize arrays filled with NaN (length n)
    ori_pts = np.full(n, np.nan)
    dist_pts = np.full(n, np.nan)
    ang_pts = np.full(n, np.nan)
    bend_pts = np.full(n, np.nan)

    # Assign segment-based values to point positions
    ori_pts[:-1] = seg_ori
    dist_pts[:-1] = seg_dist
    ang_pts[1:-1] = vert_ang
    bend_pts[1:-1] = vert_bend

    return ori_pts, ang_pts, dist_pts, bend_pts

In [ ]:
def compute_features_for_gdf(gdf, geom_col="geometry"):
    
    all_ori = []
    all_ang = []
    all_dist = []
    all_bend = []

    for geom in gdf[geom_col]:
        coords = extract_coords(geom)
        ori_pts, ang_pts, dist_pts, bend_pts = features_per_point(coords)

        all_ori.append(ori_pts)
        all_ang.append(ang_pts)
        all_dist.append(dist_pts)
        all_bend.append(bend_pts)

    gdf["orientation"] = all_ori
    gdf["angle"] = all_ang
    gdf["distance"] = all_dist
    gdf["bending"] = all_bend

    return gdf

In [ ]:
gdf_with_features_1 = compute_features_for_gdf(gdf_1)
gdf_with_features_2 = compute_features_for_gdf(gdf_2)

In [ ]:
gdf_with_features_1.iloc[4]['orientation']  #9

## Load Input Data

In [ ]:
synthetic_1 = np.load('../data/preprocessing/normalized/normalized_local_close.npy')
synthetic_2 = np.load('../data/preprocessing/normalized/normalized_local_far.npy')
original = np.load('../data/preprocessing/normalized/normalized_local_original.npy')

In [ ]:
# Create data structure of the three lines
lines_merged = [original[:15000], synthetic_1[:15000], synthetic_2[:15000]]

In [ ]:
def point_to_polyline_distance(point, polyline):
    """
    point: (2,)
    polyline: (N, 2)
    returns: shortest distance from point to any segment in the polyline
    """
    A = polyline[:-1]   # (N-1, 2)
    B = polyline[1:]    # (N-1, 2)
    AP = point - A      # (N-1, 2)
    AB = B - A          # (N-1, 2)
    AB_norm_sq = np.sum(AB**2, axis=1, keepdims=True)  # (N-1,1)
    
    t = np.clip(np.sum(AP*AB, axis=1, keepdims=True) / AB_norm_sq, 0, 1)  # (N-1,1)
    proj = A + t * AB  # projection on segment (N-1, 2)
    dist = np.linalg.norm(point - proj, axis=1)  # (N-1,)
    return np.min(dist)

In [ ]:
def compute_orientation_angles(coords):
    """
    coords: (N, 2)
    returns: (N, 1) array of angles in radians
    Angle at point i = direction from i to i+1
    For the last point, we copy the angle of the previous point.
    """
    N = len(coords)
    angles = np.zeros(N)

    for i in range(N - 1):
        dx = coords[i+1, 0] - coords[i, 0]
        dy = coords[i+1, 1] - coords[i, 1]
        angles[i] = np.arctan2(dy, dx)  # range (-pi, pi)

    # Last point: reuse previous angle
    angles[-1] = angles[-2]
    
    return angles.reshape(-1, 1)


In [ ]:
def make_graphs(lines):
    """
    lines: [normalized_original, normalized_synthetic_1, normalized_synthetic_2]
    Goal: predict shift from synthetic_1 -> synthetic_2
    """
    graphs = []
    num_lines = len(lines) #normally three
    num_seqs, seq_len, _ = lines[0].shape # seq_len: length of each polyline 

    for seq_idx in range(num_seqs):  # iterate over sequences
        node_features_list = []
        edge_list = []

        for line_id, line_array in enumerate(lines):
            coords = line_array[seq_idx]  # (seq_len, 2)

            # Compute distances to the two other lines (for features)
            other_coords = [lines[i][seq_idx] for i in range(num_lines) if i != line_id]
            #d1 = np.linalg.norm(coords - other_coords[0], axis=1, keepdims=True)
            #d2 = np.linalg.norm(coords - other_coords[1], axis=1, keepdims=True)

            #TBC: andere Distanz?
            d1 = [point_to_polyline_distance(p, other_coords[0]) for p in coords]
            d2 = [point_to_polyline_distance(p, other_coords[1]) for p in coords]
            d1 = np.array(d1).reshape(-1,1)
            d2 = np.array(d2).reshape(-1,1)
            
            #Angles and curvature 
            angles = compute_orientation_angles(coords)
            angles_norm = (angles + np.pi) / (2 * np.pi)
            #curvature = np.diff(angles, prepend=angles[0])

            # Node features: [line_id, x, y, seq_len, dist1, dist2]
            line_ids = np.full((seq_len, 1), line_id)
            seq_lengths = np.full((seq_len, 1), seq_len)
            seq_lengths_norm = seq_lengths / 64
            node_features = np.hstack([line_ids, coords, seq_lengths_norm, d1, d2, angles_norm])
            node_features_list.append(node_features)

            # --- Within-line edges ---
            start_idx = line_id * seq_len
            src = np.arange(start_idx, start_idx + seq_len - 1)
            dst = np.arange(start_idx + 1, start_idx + seq_len)
            edges = np.vstack([np.hstack([src, dst]), np.hstack([dst, src])])
            edge_list.append(edges)

        # Cross-line edges (connect same index across lines)
        for i in range(seq_len):
            for l1 in range(num_lines):
                for l2 in range(l1 + 1, num_lines):
                    n1 = l1 * seq_len + i
                    n2 = l2 * seq_len + i
                    edge_list.append(np.array([[n1, n2], [n2, n1]]))

        # Stack features and edges
        node_features = np.vstack(node_features_list) 
        edge_index = np.hstack(edge_list)              # shape (2, num_edges)

        # === TARGET: shift only for synthetic_1 nodes ===
        syn1 = lines[1][seq_idx]  # input
        syn2 = lines[2][seq_idx]  # desired output
        shift = syn2 - syn1       # Δx, Δy for each point in synthetic_1

        # Create y: zeros for other lines, shift for synthetic_1
        y = np.zeros((num_lines * seq_len, 2), dtype=np.float32)
        start_idx = 1 * seq_len
        y[start_idx:start_idx+seq_len] = shift

        # Convert to PyTorch tensors
        x = torch.tensor(node_features, dtype=torch.float)
        edge_index = torch.tensor(edge_index, dtype=torch.long)
        y = torch.tensor(y, dtype=torch.float)

        data = Data(x=x, edge_index=edge_index, y=y)
        graphs.append(data)

    return graphs


In [ ]:
graphs_seq = make_graphs(lines_merged)

In [ ]:
graphs_seq[0]

In [ ]:
def make_all_graphs_delaunay(lines):

    num_lines = len(lines)               # should be 3
    num_seqs, seq_len, _ = lines[0].shape
    graphs = []
    
    for seq_idx in range(num_seqs):
        # --- 1) Stack all points (192, 2) ---
        coords = np.vstack([lines[i][seq_idx] for i in range(num_lines)])
        
        # --- 2) Node features ---
        # line_id (192, 1)
        line_ids = np.repeat(np.arange(num_lines), seq_len).reshape(-1, 1)
        # seq_len (192, 1)
        seq_lengths = np.full((coords.shape[0], 1), seq_len)
        seq_lengths_norm = seq_lengths / 64

        # Distances to the two other lines
        dists = []
        for li in range(num_lines):
            node_coords = lines[li][seq_idx]           # (64, 2)
            others = [lines[j][seq_idx] for j in range(num_lines) if j != li]
            d1 = np.linalg.norm(node_coords - others[0], axis=1, keepdims=True)
            d2 = np.linalg.norm(node_coords - others[1], axis=1, keepdims=True)
            dists.append(np.hstack([d1, d2]))          # (64, 2)
        dists = np.vstack(dists)                       # (192, 2)

        angles = compute_orientation_angles(coords)
        angles_norm = (angles + np.pi) / (2 * np.pi)

        # Final node features: [line_id, x, y, seq_len, dist1, dist2]
        node_features = np.hstack([line_ids, coords, seq_lengths_norm, dists, angles_norm])
        x = torch.tensor(node_features, dtype=torch.float)

        # --- 3) Delaunay edges ---
        tri = Delaunay(coords)
        edges = set()
        for simplex in tri.simplices:     # 3 vertices per triangle
            for i in range(3):
                for j in range(i+1, 3):
                    a, b = simplex[i], simplex[j]
                    edges.add((a, b))
                    edges.add((b, a))     # undirected

        edge_index = torch.tensor(list(zip(*edges)), dtype=torch.long)

        # --- 4) Create Data object ---
        graphs.append(Data(x=x, edge_index=edge_index))
    
    return graphs

In [ ]:
graphs_delaunay = make_all_graphs_delaunay(lines_merged)

### Save Results

In [ ]:
torch.save(graphs_seq, f'../data/final_dataset/graph/graphs_sequential.pt')
torch.save(graphs_delaunay, f'../data/final_dataset/graph//graphs_delaunay.pt')

### Ploting Graphs

In [ ]:
def plot_graph(data):
    # Convert PyG Data -> NetworkX
    G = to_networkx(data, to_undirected=True)

    # Extract node positions (x,y from features)
    pos = {i: (float(data.x[i][1]), float(data.x[i][2])) for i in range(data.num_nodes)}

    # Node colors by line_id
    color_map = ["#143642", "#EC9A29", "#A8201A"]
    colors = [color_map[int(data.x[i][0].item())] for i in range(data.num_nodes)]

    plt.figure(figsize=(8, 6))
    plt.title(f'Constructed Graph')
    plt.xlabel('X')
    plt.ylabel('Y')
    nx.draw(G, pos,
            node_size=10,
            node_color=colors,
            edge_color="#B3B5B6A6",
            alpha=0.8)
    plt.show()

In [ ]:
graph = graphs_delaunay   
g0 = graph[0]       

df = pd.DataFrame(
    g0.x.numpy(),
    columns=["line_id", "x", "y", "seq_len", "dist1", "dist2", "angle"]
)
#print(df.head())
print(df[df["line_id"] == 2].head())


In [ ]:
plot_graph(g0)